# Download and Prepare the Archived Cluebase Data

This notebook creates local CSV files from the archived Cluebase PostgreSQL database.

The API hostname currently does not resolve, which produces a `NameResolutionError`. Retrying the request or changing the pagination settings cannot fix a missing DNS record. Instead, this notebook downloads the database snapshot stored in the official Cluebase GitHub repository.

## What the notebook produces

- One CSV for every table found in the Cluebase database
- `cluebase_contestant_games.csv`: one row per contestant per game
- `cluebase_game_boards.csv`: one row per game with combined clue text
- `cluebase_modeling_index.csv`: contestant-game rows joined to game-board text
- `cluebase_csv_files.zip`: a convenient archive of all generated CSV files

The database snapshot is from 2019. When joining it to a newer scoring dataset, retain only games present in both sources.

## 1. Import packages and choose output locations

The notebook stores the downloaded SQL file separately from the generated CSV files. Re-running the download cell will reuse a completed SQL file unless `DOWNLOAD_AGAIN` is changed to `True`.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys

import pandas as pd
import requests

SQL_URL = (
    "https://raw.githubusercontent.com/lukelavin/Cluebase/"
    "master/postgres/init/jeopardy201908021145.sql"
)

PROJECT_DIR = Path("cluebase_project")
RAW_DIR = PROJECT_DIR / "raw"
CSV_DIR = PROJECT_DIR / "csv"
SQL_PATH = RAW_DIR / "cluebase_database.sql"
MANIFEST_PATH = PROJECT_DIR / "download_manifest.json"

RAW_DIR.mkdir(parents=True, exist_ok=True)
CSV_DIR.mkdir(parents=True, exist_ok=True)

DOWNLOAD_AGAIN = False

print("Project folder:", PROJECT_DIR.resolve())
print("CSV folder:", CSV_DIR.resolve())

Project folder: /content/cluebase_project
CSV folder: /content/cluebase_project/csv


## 2. Download the archived database

The file is downloaded in chunks so it does not need to be held entirely in memory. A SHA-256 checksum and retrieval time are recorded in a small manifest for reproducibility. GitHub may report the compressed transfer size in the `Content-Length` header while `requests` saves decompressed bytes, so the notebook validates the downloaded content rather than requiring those two sizes to match.

In [ ]:
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


expected_bytes = None

if DOWNLOAD_AGAIN or not SQL_PATH.exists() or SQL_PATH.stat().st_size == 0:
    temporary_path = SQL_PATH.with_suffix(".partial")

    with requests.get(SQL_URL, stream=True, timeout=180) as response:
        response.raise_for_status()
        expected_bytes = int(response.headers.get("content-length", 0))

        with temporary_path.open("wb") as file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file.write(chunk)

    temporary_path.replace(SQL_PATH)

else:
    print("Using the SQL file that is already present.")

actual_bytes = SQL_PATH.stat().st_size
with SQL_PATH.open("rb") as file:
    opening_bytes = file.read(4096)

if actual_bytes < 1_000_000:
    raise IOError(
        f"The downloaded file is unexpectedly small ({actual_bytes} bytes). "
        "Delete it and rerun this cell."
    )

if b"PostgreSQL database dump" not in opening_bytes:
    raise IOError(
        "The downloaded file does not look like a PostgreSQL database dump. "
        "Delete it and rerun this cell."
    )

manifest = {
    "source_url": SQL_URL,
    "retrieved_utc": datetime.now(timezone.utc).isoformat(),
    "file_name": SQL_PATH.name,
    "file_size_bytes": actual_bytes,
    "server_content_length_header_bytes": expected_bytes,
    "sha256": sha256_file(SQL_PATH),
}

MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(f"SQL file size: {actual_bytes / 1_000_000:.1f} MB")
print("SHA-256:", manifest["sha256"])

Using the SQL file that is already present.
SQL file size: 50.4 MB
SHA-256: d1d3cb6b169652ad48de490042b953bc17309037ecf56250bf366e7ec654156c


## 3. Install and start PostgreSQL

Google Colab normally runs with the permissions needed for this installation. If PostgreSQL is already available, the installation commands are skipped.

This notebook resets only a local temporary database named `jeopardy`. It does not change the downloaded SQL file.

In [ ]:
def run(command, **kwargs):
    print("Running:", " ".join(map(str, command)))
    return subprocess.run(command, check=True, **kwargs)


if shutil.which("psql") is None:
    if hasattr(os, "geteuid") and os.geteuid() != 0:
        raise PermissionError(
            "PostgreSQL is not installed and this notebook is not running as root. "
            "Run the notebook in Google Colab or install PostgreSQL manually."
        )
    run(["apt-get", "-qq", "update"])
    run(["apt-get", "-qq", "install", "-y", "postgresql"])

run(["service", "postgresql", "start"])
print("PostgreSQL is ready.")

Running: service postgresql start
PostgreSQL is ready.


## 4. Load the SQL snapshot

The following cell recreates the local `jeopardy` database and loads the archived schema and records. The detailed PostgreSQL output is written to `cluebase_load_log.txt` rather than filling the notebook.

In [ ]:
def postgres_command(arguments):
    # Run a command as the local postgres operating-system user.
    if shutil.which("sudo"):
        return ["sudo", "-u", "postgres", *arguments]
    if shutil.which("runuser"):
        return ["runuser", "-u", "postgres", "--", *arguments]
    raise RuntimeError("Neither sudo nor runuser is available.")


run(postgres_command(["dropdb", "--if-exists", "jeopardy"]))
run(postgres_command(["createdb", "jeopardy"]))

load_log = PROJECT_DIR / "cluebase_load_log.txt"
with load_log.open("w", encoding="utf-8") as log_file:
    load_result = subprocess.run(
        postgres_command([
            "psql",
            "-v", "ON_ERROR_STOP=1",
            "-d", "jeopardy",
            "-f", str(SQL_PATH.resolve()),
        ]),
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True,
    )

if load_result.returncode != 0:
    print(load_log.read_text(encoding="utf-8")[-4000:])
    raise RuntimeError("The database did not load. See the end of the log above.")

print("Database loaded successfully.")
print("Load log:", load_log.resolve())

Running: sudo -u postgres dropdb --if-exists jeopardy
Running: sudo -u postgres createdb jeopardy
Database loaded successfully.
Load log: /content/cluebase_project/cluebase_load_log.txt


## 5. Export every database table to CSV

The code discovers the table names automatically instead of assuming that they are singular or plural. Each table is then exported with a header row.

In [ ]:
table_query = '''
SELECT tablename
FROM pg_tables
WHERE schemaname = 'public'
ORDER BY tablename;
'''

table_result = subprocess.check_output(
    postgres_command(["psql", "-d", "jeopardy", "-Atc", table_query]),
    text=True,
)

tables = [name.strip() for name in table_result.splitlines() if name.strip()]
if not tables:
    raise RuntimeError("No public tables were found in the loaded database.")

print("Tables found:", tables)

for table in tables:
    if not re.fullmatch(r"[A-Za-z0-9_]+", table):
        raise ValueError(f"Unexpected table name: {table}")

    csv_path = CSV_DIR / f"{table}.csv"
    copy_statement = f"\\copy public.{table} TO STDOUT WITH (FORMAT CSV, HEADER TRUE)"

    with csv_path.open("w", encoding="utf-8", newline="") as csv_file:
        export_result = subprocess.run(
            postgres_command([
                "psql", "-q", "-d", "jeopardy", "-c", copy_statement
            ]),
            stdout=csv_file,
            stderr=subprocess.PIPE,
            text=True,
        )

    if export_result.returncode != 0:
        raise RuntimeError(
            f"Could not export {table}: {export_result.stderr[-1000:]}"
        )

    print(f"Saved {csv_path.name}")

Tables found: ['clues', 'contestants', 'games', 'parsed_games', 'seasons']
Saved clues.csv
Saved contestants.csv
Saved games.csv
Saved parsed_games.csv
Saved seasons.csv


## 6. Load and identify the important tables

The three tables needed for this project are identified by their required columns:

- **Clues:** `id`, `game_id`, clue text, and response
- **Games:** game ID, three contestant IDs, winner ID, and final scores
- **Contestants:** contestant ID, name, and introduction notes

The introduction notes contain the occupation and hometown language that will later be mapped to SOC occupations.

In [ ]:
dataframes = {}

for csv_path in sorted(CSV_DIR.glob("*.csv")):
    frame = pd.read_csv(csv_path, low_memory=False)
    frame.columns = [str(column).strip().lower() for column in frame.columns]
    dataframes[csv_path.stem] = frame

summary = pd.DataFrame(
    [
        {
            "table": name,
            "rows": len(frame),
            "columns": len(frame.columns),
            "column_names": ", ".join(frame.columns),
        }
        for name, frame in dataframes.items()
    ]
).sort_values("table")

display(summary)


def find_table(required_columns, label):
    required_columns = set(required_columns)
    matches = [
        (name, frame)
        for name, frame in dataframes.items()
        if required_columns.issubset(set(frame.columns))
    ]
    if len(matches) != 1:
        candidates = [name for name, _ in matches]
        raise RuntimeError(
            f"Expected one {label} table but found {len(matches)}: {candidates}. "
            "Review the table summary and select the correct table manually."
        )
    return matches[0]


clue_text_column = None
for candidate in ("clue", "question"):
    possible = [
        (name, frame)
        for name, frame in dataframes.items()
        if {"id", "game_id", candidate}.issubset(frame.columns)
    ]
    if possible:
        clue_text_column = candidate
        break

if clue_text_column is None:
    raise RuntimeError("A clue-text column named 'clue' or 'question' was not found.")

clues_name, clues = find_table(
    {"id", "game_id", clue_text_column}, "clue"
)
games_name, games = find_table(
    {"id", "contestant1", "contestant2", "contestant3", "winner"}, "game"
)
contestants_name, contestants = find_table(
    {"id", "name", "notes"}, "contestant"
)

print("Clues table:", clues_name, clues.shape)
print("Games table:", games_name, games.shape)
print("Contestants table:", contestants_name, contestants.shape)

,table,rows,columns,column_names
0,cluebase_contestant_games,18924,14,"game_id, contestant_position, contestant_id, f..."
1,cluebase_game_boards,6304,4,"game_id, clue_count, board_text, category_count"
2,cluebase_modeling_index,18924,17,"game_id, contestant_position, contestant_id, f..."
3,clues,366556,8,"id, game_id, value, daily_double, round, categ..."
4,contestants,12028,5,"id, name, notes, games_played, total_winnings"
5,games,6308,12,"id, episode_num, season_id, air_date, notes, c..."
6,parsed_games,6377,3,"id, episode_num, game_link"
7,seasons,36,5,"id, season_name, start_date, end_date, total_g..."


Clues table: clues (366556, 8)
Games table: games (6308, 12)
Contestants table: contestants (12028, 5)


## 7. Create one contestant-game row for each player

The original game table stores the three contestants in separate columns. Modeling is easier when each contestant occupies a separate row. The resulting table includes the contestant's game position, final score, winner indicator, name, and introduction notes.

In [ ]:
game_metadata_columns = [
    column
    for column in ["id", "episode_num", "season_id", "air_date", "notes"]
    if column in games.columns
]

contestant_game_records = []

for game in games.to_dict(orient="records"):
    for position in (1, 2, 3):
        contestant_id = game.get(f"contestant{position}")
        final_score = game.get(f"score{position}")

        record = {
            "game_id": game["id"],
            "contestant_position": position,
            "contestant_id": contestant_id,
            "final_score": final_score,
            "winner_id": game.get("winner"),
            "is_winner": int(
                pd.notna(contestant_id)
                and pd.notna(game.get("winner"))
                and contestant_id == game.get("winner")
            ),
        }

        for column in game_metadata_columns:
            if column == "id":
                continue
            output_name = "game_notes" if column == "notes" else column
            record[output_name] = game.get(column)

        contestant_game_records.append(record)

contestant_games = pd.DataFrame(contestant_game_records)

contestant_lookup = contestants.rename(
    columns={
        "id": "contestant_id",
        "name": "contestant_name",
        "notes": "contestant_notes",
    }
)

keep_contestant_columns = [
    column
    for column in [
        "contestant_id",
        "contestant_name",
        "contestant_notes",
        "games_played",
        "total_winnings",
    ]
    if column in contestant_lookup.columns
]

contestant_games = contestant_games.merge(
    contestant_lookup[keep_contestant_columns],
    on="contestant_id",
    how="left",
    validate="many_to_one",
)

contestant_games_path = CSV_DIR / "cluebase_contestant_games.csv"
contestant_games.to_csv(contestant_games_path, index=False)

print("Saved:", contestant_games_path)
print("Shape:", contestant_games.shape)
display(contestant_games.head())

Saved: cluebase_project/csv/cluebase_contestant_games.csv
Shape: (18924, 14)


,game_id,contestant_position,contestant_id,final_score,winner_id,is_winner,episode_num,season_id,air_date,game_notes,contestant_name,contestant_notes,games_played,total_winnings
0,336,1,640,14601,640,1,7715,2,2018-03-09,NaN,Lane Flynn,"a business owner from Atlanta, Georgia",4,40399
1,336,2,641,7400,640,0,7715,2,2018-03-09,NaN,Megan Durazo,"a librarian from Playa Del Rey, California",2,1000
2,336,3,638,29100,640,0,7715,2,2018-03-09,NaN,Mark Ashton,"a software engineer from Elmhurst, Illinois",3,60200
3,337,1,640,19201,643,0,7714,2,2018-03-08,NaN,Lane Flynn,"a business owner from Atlanta, Georgia",4,40399
4,337,2,642,9601,643,0,7714,2,2018-03-08,NaN,Hannah Ewing,"a teacher from Stamford, Connecticut",1,1000


## 8. Create one combined clue board per game

The board table keeps the original game ID and combines each category with its clue text. This makes it possible to compare a contestant occupation profile with the language of the board using TF-IDF or Sentence-BERT.

In [ ]:
clue_work = clues.copy()
clue_work["game_id"] = pd.to_numeric(clue_work["game_id"], errors="coerce")
clue_work[clue_text_column] = clue_work[clue_text_column].fillna("").astype(str)

if "category" not in clue_work.columns:
    clue_work["category"] = ""
else:
    clue_work["category"] = clue_work["category"].fillna("").astype(str)

clue_work["clue_with_category"] = (
    clue_work["category"].str.strip()
    + ": "
    + clue_work[clue_text_column].str.strip()
).str.strip(": ")

aggregation = {
    "clue_count": ("id", "count"),
    "board_text": (
        "clue_with_category",
        lambda values: " ".join(value for value in values if value),
    ),
}

if "category" in clue_work.columns:
    aggregation["category_count"] = ("category", "nunique")

game_boards = (
    clue_work.dropna(subset=["game_id"])
    .groupby("game_id", as_index=False)
    .agg(**aggregation)
)

game_boards_path = CSV_DIR / "cluebase_game_boards.csv"
game_boards.to_csv(game_boards_path, index=False)

modeling_index = contestant_games.merge(
    game_boards,
    on="game_id",
    how="left",
    validate="many_to_one",
)

modeling_index_path = CSV_DIR / "cluebase_modeling_index.csv"
modeling_index.to_csv(modeling_index_path, index=False)

print("Saved:", game_boards_path)
print("Saved:", modeling_index_path)
display(game_boards.head())

Saved: cluebase_project/csv/cluebase_game_boards.csv
Saved: cluebase_project/csv/cluebase_modeling_index.csv


,game_id,clue_count,board_text,category_count
0,1,60,VENICE: The name of this bridge is a contracti...,12
1,2,54,"TV SPINOFFS: An episode of ""Happy Days"" in whi...",12
2,3,60,KING JAMES BIBLE BIRDWATCHING: In John 1:3 the...,12
3,4,60,"BRITISH LITERATURE: Better known for this ""Tal...",12
4,5,59,audible SUMMER READING: Ian McKellen narrates ...,12


## 9. Run quality checks

These checks do not automatically delete records. They show whether the expected three-contestant structure is present, whether each game has exactly one labeled winner, how often contestant introductions are missing, and how many clues were matched to each game.

In [ ]:
contestants_per_game = contestant_games.groupby("game_id")["contestant_id"].count()
winners_per_game = contestant_games.groupby("game_id")["is_winner"].sum()

quality_summary = pd.Series({
    "games_in_game_table": games["id"].nunique(),
    "games_in_contestant_game_table": contestant_games["game_id"].nunique(),
    "contestant_game_rows": len(contestant_games),
    "games_with_exactly_3_contestants": int((contestants_per_game == 3).sum()),
    "games_with_exactly_1_winner": int((winners_per_game == 1).sum()),
    "rows_missing_contestant_name": int(contestant_games["contestant_name"].isna().sum()),
    "rows_missing_contestant_notes": int(contestant_games["contestant_notes"].isna().sum()),
    "games_with_clue_board_text": int(game_boards["board_text"].notna().sum()),
})

display(quality_summary.to_frame("value"))

print("Contestants per game")
display(contestants_per_game.value_counts(dropna=False).sort_index().to_frame("games"))

print("Winner labels per game")
display(winners_per_game.value_counts(dropna=False).sort_index().to_frame("games"))

print("Clues per game")
display(game_boards["clue_count"].describe().to_frame())

,value
games_in_game_table,6308
games_in_contestant_game_table,6308
contestant_game_rows,18924
games_with_exactly_3_contestants,6308
games_with_exactly_1_winner,6308
rows_missing_contestant_name,0
rows_missing_contestant_notes,0
games_with_clue_board_text,6304


Contestants per game


,games
contestant_id,
3,6308


Winner labels per game


,games
is_winner,
1,6308


Clues per game


,clue_count
count,6304.000000
mean,58.146574
std,3.432414
min,7.000000
25%,57.000000
50%,60.000000
75%,60.000000
max,60.000000


## 10. Save a ZIP archive

The ZIP file contains all raw table exports and the three project-ready CSVs. In Google Colab, uncomment the final two lines if you want the browser to download the ZIP automatically.

In [ ]:
archive_base = PROJECT_DIR / "cluebase_csv_files"
archive_path = shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=CSV_DIR,
)

print("ZIP archive:", archive_path)

# Optional automatic browser download in Google Colab:
# from google.colab import files
# files.download(archive_path)

ZIP archive: /content/cluebase_project/cluebase_csv_files.zip


## Joining this data to the scoring dataset

Use a hierarchical matching process rather than assuming that Cluebase's internal `game_id` is shared by another dataset.

1. Standardize the external scoring dataset's episode or show number and air date.
2. First attempt an exact match using the Cluebase `episode_num` and `air_date` fields when both are available.
3. Confirm that every selected match represents only one game in each source.
4. Match contestant names within the matched game after normalizing capitalization, punctuation, spacing, and suffixes.
5. Preserve fields such as `game_match_status`, `contestant_match_status`, and `match_confidence` instead of silently accepting uncertain matches.
6. Restrict the analysis to the intersection of the Cluebase snapshot and the scoring dataset.

The `contestant_notes` field is the source for occupation extraction. It often includes both occupation and hometown, so occupation phrases should be parsed and then reviewed before SOC mapping.

## Important limitations

- The live Cluebase API is unavailable when its hostname does not resolve.
- The GitHub SQL file is a fixed 2019 snapshot and will not contain later games.
- Cluebase data were scraped from J-Archive and may contain missing or incorrect records.
- J! Archive currently prohibits automated scraping and systematic collection in its Terms of Use, so this notebook does not scrape the site directly.
- `game_id` is an internal Cluebase key and should not be assumed to match IDs from another source.
- Tournament games, games with missing contestants, and games without exactly one winner require explicit inclusion rules.
- Keep the original SQL file and raw CSV exports unchanged. Perform cleaning in new files.

## Sources

- Cluebase repository: https://github.com/lukelavin/Cluebase
- Archived SQL database: https://github.com/lukelavin/Cluebase/blob/master/postgres/init/jeopardy201908021145.sql
- Cluebase documentation: https://cluebase.readthedocs.io/en/latest/
- J! Archive Terms of Use: https://www.j-archive.com/help.php#terms

## Troubleshooting

**`NameResolutionError` for `cluebase.lukelav.in`**  
The former API address is not reachable. Use the GitHub SQL workflow in this notebook.

**The SQL download stops early**  
Delete `cluebase_project/raw/cluebase_database.sql` and any `.partial` file, then rerun the download cell. The notebook verifies that the result has the expected PostgreSQL dump signature and a plausible minimum size.

**The saved file is larger than the `Content-Length` header**  
This is expected when GitHub sends compressed content and `requests` automatically decompresses it. The corrected notebook checks the SQL signature and a minimum plausible size instead of requiring the two byte counts to match.

**PostgreSQL installation fails**  
Open the notebook in Google Colab and restart the runtime. A restricted school or local Jupyter environment may not allow operating-system package installation.

**The database load fails**  
Read the final lines of `cluebase_project/cluebase_load_log.txt`. Restarting the runtime and running the notebook from the beginning usually provides a clean PostgreSQL instance.

**A required table is not detected**  
Review the table summary produced in Section 6. The notebook deliberately stops rather than guessing when multiple tables match the same required columns.

In [ ]:
from pathlib import Path
from io import StringIO
import pandas as pd
import re
import csv

# Locate the downloaded Cluebase PostgreSQL dump
if "SQL_PATH" in globals() and Path(SQL_PATH).exists():
    sql_path = Path(SQL_PATH)
else:
    sql_files = list(Path("/content").rglob("*.sql"))

    if not sql_files:
        raise FileNotFoundError("No .sql file was found in /content.")

    # The database dump should be the largest SQL file
    sql_path = max(sql_files, key=lambda path: path.stat().st_size)

print("Reading:", sql_path)
print("File size:", round(sql_path.stat().st_size / 1_000_000, 2), "MB")

copy_pattern = re.compile(
    r'^COPY\s+(?:public\.)?"?([^"\s.]+)"?\s+\((.*?)\)\s+FROM\s+stdin;$',
    re.IGNORECASE
)

game_df = None

with sql_path.open("r", encoding="utf-8", errors="replace") as sql_file:
    for line in sql_file:
        match = copy_pattern.match(line.strip())

        if not match:
            continue

        table_name = match.group(1)
        columns = [
            column.strip().strip('"')
            for column in match.group(2).split(",")
        ]

        # Identify the table containing game, season, and date information
        if "season_id" in columns and "air_date" in columns:
            rows = []

            for data_line in sql_file:
                if data_line.rstrip("\n") == r"\.":
                    break
                rows.append(data_line)

            game_df = pd.read_csv(
                StringIO("".join(rows)),
                sep="\t",
                names=columns,
                header=None,
                dtype=str,
                na_values=[r"\N"],
                keep_default_na=False,
                quoting=csv.QUOTE_NONE
            )

            print("Game table found:", table_name)
            break

if game_df is None:
    raise ValueError(
        "A table containing season_id and air_date was not found in the SQL dump."
    )

# Convert dates and summarize seasons
game_df["air_date"] = pd.to_datetime(
    game_df["air_date"],
    errors="coerce"
)

season_summary = (
    game_df.groupby("season_id", dropna=False)
    .agg(
        number_of_games=("id", "nunique"),
        first_air_date=("air_date", "min"),
        last_air_date=("air_date", "max")
    )
    .reset_index()
)

season_summary["season_number"] = pd.to_numeric(
    season_summary["season_id"],
    errors="coerce"
)

season_summary = (
    season_summary
    .sort_values("season_number")
    .drop(columns="season_number")
)

display(season_summary)

print("Number of seasons:", season_summary["season_id"].nunique())
print("Earliest game:", game_df["air_date"].min().date())
print("Latest game:", game_df["air_date"].max().date())

Reading: cluebase_project/raw/cluebase_database.sql
File size: 50.42 MB
Game table found: games


,season_id,number_of_games,first_air_date,last_air_date
0,1,230,2018-09-10,2019-07-26
11,2,230,2017-09-11,2018-07-27
22,3,230,2016-09-12,2017-07-28
30,4,230,2015-09-14,2016-07-29
31,5,230,2014-09-15,2015-07-31
32,6,230,2013-09-16,2014-08-01
33,7,230,2012-09-17,2013-08-02
34,8,230,2011-09-19,2012-08-03
35,9,229,2010-09-13,2011-07-29
1,10,230,2009-09-14,2010-07-30


Number of seasons: 36
Earliest game: 1984-09-27
Latest game: 2019-07-26
